# Cola del ingreso corriente de hogares: ENIGH 2024

Estudio **exploratorio** con la tabla `CONCENTRADOHOGAR` de la [ENIGH 2024, nueva serie](https://www.inegi.org.mx/programas/enigh/nc/2024/). Descarga los microdatos CSV oficiales y coloca `concentradohogar.csv` en `examples/data/`; ejecuta desde `examples/`. Consulta la [descripción de la base](https://www.inegi.org.mx/contenidos/programas/enigh/nc/2024/microdatos/889463924494.pdf) y el [cálculo oficial en R](https://www.inegi.org.mx/contenidos/programas/enigh/nc/2024/microdatos/889463924845.pdf). Instala `pip install -e ".[analysis]"` desde la raíz del proyecto.

`ing_cor` es ingreso corriente trimestral del hogar y `factor` es su factor de expansión. `powerlaw.Fit` no acepta pesos muestrales. El ajuste siguiente describe **hogares observados sin ponderar** y no estima la distribución nacional ni su incertidumbre bajo el diseño muestral de ENIGH.


In [ ]:
from hashlib import sha256
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd

from economista.analysis import fit_power_law

DATA_PATH = Path("data/concentradohogar.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Descarga el CSV oficial en {DATA_PATH}")
print("SHA256:", sha256(DATA_PATH.read_bytes()).hexdigest())
print("economista:", version("economista"), "powerlaw:", version("powerlaw"))


## Limpieza y descripción ponderada

Se conservan hogares con ingreso y factor positivos. La curva ponderada usa `factor` sólo como descripción; no altera el ajuste posterior. Se informan las exclusiones.


In [ ]:
raw = pd.read_csv(
    DATA_PATH, encoding="latin-1",
    usecols=["ing_cor", "factor"], low_memory=False,
)
for column in ("ing_cor", "factor"):
    raw[column] = pd.to_numeric(raw[column], errors="coerce")
homes = raw.loc[raw["ing_cor"].gt(0) & raw["factor"].gt(0)].copy()
print({"hogares_originales": len(raw), "hogares_validos": len(homes),
       "excluidos": len(raw) - len(homes)})

sorted_homes = homes.sort_values("ing_cor")
weighted_tail = (
    sorted_homes["factor"].iloc[::-1].cumsum().iloc[::-1]
    / sorted_homes["factor"].sum()
)
observed_tail = np.arange(len(sorted_homes), 0, -1) / len(sorted_homes)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(sorted_homes["ing_cor"], observed_tail, label="Muestra observada")
ax.loglog(sorted_homes["ing_cor"], weighted_tail, label="Descripción con factor")
ax.set(xlabel="Ingreso corriente trimestral por hogar (MXN)", ylabel="P(X ≥ x)",
       title="ENIGH 2024: descripción de cola")
ax.legend()
plt.show()


## Ajuste exploratorio sin ponderación

Fijamos el percentil 90 de la muestra observada como umbral preespecificado para comparar la cola superior. No se replica la estimación oficial del ingreso nacional. La comparación con lognormal se realiza sobre la misma muestra sin ponderar.


In [ ]:
xmin = float(homes["ing_cor"].quantile(0.90))
fit = fit_power_law(homes, column="ing_cor", xmin=xmin)
comparison = fit.compare_to("lognormal")
print({"alpha": fit.alpha, "xmin_mxn_trimestral": fit.xmin,
       "ks_distance": fit.ks_distance, "sigma": fit.sigma, "n_tail": fit.n_tail})
print({"alternativa": comparison.alternative,
       "log_likelihood_ratio": comparison.log_likelihood_ratio,
       "p_value": comparison.p_value})


## Interpretación y límites

Antes de publicar, registra hash, fecha de descarga, exclusiones, umbral y sensibilidad a otros percentiles. No interpretes `alpha`, `sigma`, la razón de verosimilitud ni su valor p como inferencia nacional: el muestreo de ENIGH es estratificado y por conglomerados. La curva con `factor` tampoco sustituye una estimación de incertidumbre que considere el diseño. Para afirmaciones poblacionales hace falta una metodología muestral específica.
